In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer

dataset_candidates = [
    Path(r"C:\Users\happy\OneDrive\dataset\uci-secom.csv"),
    Path("uci-secom.csv"),
]
csv_path = next((path for path in dataset_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("uci-secom.csv not found in dataset folder or project root")

print("Loading VectraYield from:", csv_path)
vectrayield = pd.read_csv(csv_path)
vectrayield.sample(10)

In [ ]:
vectrayield.shape

In [ ]:
vectrayield.info()

In [ ]:
print("Total Missing Values",vectrayield.isnull().sum().sum())

In [ ]:
print("Total duplicate values",vectrayield.duplicated().sum().sum())

In [ ]:
vectrayield_missing=vectrayield.isnull().mean()
print(vectrayield_missing*100)

In [ ]:
vectrayield_HighNulls = vectrayield_missing[vectrayield_missing >= 0.50]
print("Columns with missing values >= 50%")
print(vectrayield_HighNulls)
print("Count:", len(vectrayield_HighNulls))

In [ ]:
num_cols = vectrayield.select_dtypes(include=['float64', 'int64']).columns

outlier_info = {}

for col in num_cols:
    Q1 = vectrayield[col].quantile(0.25)
    Q3 = vectrayield[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = vectrayield[(vectrayield[col] < lower_bound) | (vectrayield[col] > upper_bound)]
    
    if len(outliers) > 0:
        outlier_info[col] = len(outliers)
print("columns with outliers count is ",len(outlier_info))


## Contribution: missing-value handling and outlier capping

Drop columns with 50%+ missing values, fill remaining numeric gaps with SimpleImputer(strategy="median"), then cap outliers with IQR trimming (winsorization).

In [ ]:
vectrayield_clean = vectrayield.drop(columns=vectrayield_HighNulls.index, errors="ignore")
print("Shape after dropping high-null columns:", vectrayield_clean.shape)

In [ ]:
target_col = "Pass/Fail"
time_col = "Time" if "Time" in vectrayield_clean.columns else None

feature_cols = [c for c in vectrayield_clean.columns if c not in {target_col, time_col}]
numeric_cols = vectrayield_clean[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

imputer = SimpleImputer(strategy="median")
vectrayield_imputed = vectrayield_clean.copy()
vectrayield_imputed[numeric_cols] = imputer.fit_transform(vectrayield_clean[numeric_cols])

print("Missing values after SimpleImputer:", int(vectrayield_imputed[numeric_cols].isnull().sum().sum()))

In [ ]:
def iqr_cap(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower=lower, upper=upper)

vectrayield_capped = vectrayield_imputed.copy()
vectrayield_capped[numeric_cols] = vectrayield_imputed[numeric_cols].apply(iqr_cap)

print("Shape after IQR trimming/capping:", vectrayield_capped.shape)
print("Remaining missing values:", int(vectrayield_capped.isnull().sum().sum()))
vectrayield_capped.head()